<a href="https://colab.research.google.com/github/mastercljohnson/ppo_impl/blob/main/ppo_raw_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gymnasium[mujoco] mediapy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 38.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 32.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [mediapy]


In [ ]:
from numpy._core.fromnumeric import std
import torch
import torch.nn as nn

class Policy(nn.Module):
  def __init__(self):
    super(Policy, self).__init__()
    self.fc1 = nn.Linear(17, 64)
    self.relu = nn.ReLU()                          # Activation function
    self.fc2 = nn.Linear(64, 6)
    # self.tanh = nn.Tanh()
    # Standard deviation must also be trainable for this to converge in practice
    self.log_var = nn.Parameter(torch.zeros(6))

  def forward(self, x):
        # Define the forward pass (data flow)
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        # out = self.tanh(out)
        return out

  def get_distribution(self, x):
    action_means = self.forward(x)
    var = torch.exp(self.log_var)
    cov_matrix = torch.diag(var)
    return torch.distributions.MultivariateNormal(loc=action_means, covariance_matrix=cov_matrix)


class Value(nn.Module):
  def __init__(self):
    super(Value, self).__init__()
    self.fc1 = nn.Linear(17, 64)
    self.relu = nn.ReLU()                          # Activation function
    self.fc2 = nn.Linear(64, 1)

  def forward(self, x):
      # Define the forward pass (data flow)
      out = self.fc1(x)
      out = self.relu(out)
      out = self.fc2(out)
      return out

In [ ]:
# 2. Import and create the HalfCheetah environment
import os
import gymnasium as gym
import mediapy as media

# 1. Set the environment variable to use EGL for headless rendering
os.environ["MUJOCO_GL"] = "egl"

env = gym.make("HalfCheetah-v5", render_mode="rgb_array")

def rollout(env, policy, horizon, render):
  frames = []

  states, zs ,actions, log_probs, rewards, dones = [], [], [], [], [], []

  device = next(policy.parameters()).device

  state, info = env.reset()

  for i in range(horizon):

    state_tensor = torch.from_numpy(state).float().to(device)

    with torch.no_grad():
      act_dist = policy.get_distribution(state_tensor)
      #  Try using tanh gaussian for performance instead of clamping actions for half cheetah action space
      z = act_dist.rsample()
      action_tensor = torch.tanh(z)

      # action_tensor = torch.clamp(action_tensor, -1.0, 1.0)
      log_prob = act_dist.log_prob(z)
      log_prob -= torch.log(1- torch.tanh(z).pow(2) + 1e-6).sum(-1)

      action = action_tensor.cpu().numpy()

    next_state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated

    # Reward hack to break out of the scoot on back solution :)
    # This creates a reward cliff, critic is unable to stabilize since reward isnt smooth
    # torso_angle = state[1]
    # if abs(torso_angle) > 1.0:
    #   reward -= 10.0

    if render:
      frames.append(env.render())

    states.append(state)
    zs.append(z.detach().cpu())
    actions.append(action)
    log_probs.append(log_prob.cpu())
    rewards.append(reward)
    dones.append(terminated or truncated)

    if done:
      state, info = env.reset()
    else:
      state = next_state

  # final state transition
  states.append(state)

  # pre_clamp = np.array(pre_clamp)

  # print(f"action tensor mean {pre_clamp.mean().item()}, min {pre_clamp.min()}, max{pre_clamp.max()}")

  return states, zs, actions, log_probs, rewards, dones, frames





In [ ]:
import numpy as np

# General Advantage Estimation
def adv(critic, states, rewards, dones, gamma, lam):

  if isinstance(states, list):
        states = torch.from_numpy(np.asarray(states)).float()
  if isinstance(rewards, list):
        rewards = torch.tensor(rewards, dtype=torch.float32)
  if isinstance(dones, list):
    dones = torch.tensor(dones, dtype=torch.float32)

  masks = 1.0 - dones

  with torch.no_grad():
    v_state = critic(states[:-1]).squeeze(-1)
    v_next_state = critic(states[1:]).squeeze(-1)

    residuals = rewards + gamma * masks * v_next_state - v_state

    advantages = torch.zeros_like(rewards)
    gae = 0.0
    for t in reversed(range(len(residuals))):
      gae = residuals[t] + gamma * lam * gae * masks[t]
      advantages[t] = gae

  return advantages


In [ ]:
def clipped_obj(policy,critic, states, zs, log_probs, epsilon, advantages, ent_coeff):
  device = next(policy.parameters()).device
  states = torch.tensor(np.array(states), dtype=torch.float32).to(device)
  # zs = torch.tensor(np.array(zs), dtype=torch.float32).to(device)
  # log_probs = torch.tensor(np.array(log_probs), dtype=torch.float32).to(device)
  zs = torch.stack(zs).to(device)
  log_probs = torch.stack(log_probs).to(device)

  act_dist = policy.get_distribution(states[:-1])
  # policy_log_prob = act_dist.log_prob(actions)
  # log_ratio = policy_log_prob - log_probs

  # Recover the actions from the tanh gaussian trick
  gaussian_log_prob = act_dist.log_prob(zs) # get_dist uses multivariate norm
  log_correction = torch.log(1 - torch.tanh(zs).pow(2) + 1e-6).sum(dim=-1)
  new_log_probs = gaussian_log_prob - log_correction

  log_ratio = new_log_probs - log_probs
  prob_ratio = torch.exp(log_ratio)
  clipped_ratio = torch.clamp(prob_ratio, 1 - epsilon, 1 + epsilon)

  # Entropy of the current pre-tanh Gaussian
  gaussian_entropy = act_dist.entropy()
  # Fresh samples from the current policy, with gradients
  z_entropy = act_dist.rsample()
  # Tanh Jacobian correction, fresh samples separate from policy obj
  log_correction_entropy = torch.log(
      1 - torch.tanh(z_entropy).pow(2) + 1e-6
  ).sum(dim=-1)
  # Estimate entropy of the bounded action distribution
  entropy = (
      gaussian_entropy + log_correction_entropy
  ).mean()

  objective = torch.min(prob_ratio * advantages, clipped_ratio * advantages)
  # min -obj is the same as max obj
  loss = -objective.mean() - ent_coeff * entropy

  # ratio mean, clip fraction + KL stats for logging and debug
  mean_ratio = prob_ratio.mean().item()
  clip_fraction = ((prob_ratio - clipped_ratio).abs() > 1e-8).float().mean().item()

  # Forward KL Monte Carlo estimate
  approx_kl = (-log_ratio).mean().item()
  # Nonnegative KL diagnostic
  kl_estimate = (prob_ratio - 1 - log_ratio).mean().item()

  return loss, mean_ratio, clip_fraction, approx_kl, kl_estimate

In [ ]:
def calculate_episode_returns(rewards, dones):
    episode_returns = []
    episode_lengths = []

    current_return = 0.0
    current_length = 0

    for reward, done in zip(rewards, dones):
        current_return += reward
        current_length += 1

        if done:
            episode_returns.append(current_return)
            episode_lengths.append(current_length)

            current_return = 0.0
            current_length = 0

    return episode_returns, episode_lengths

training_log = {
    "episode_return": [],
    "episode_length": [],
    "policy_loss": [],
    "critic_loss": [],
    "mean_advantage": [],
    "std_advantage": [],
    "mean_return": [],
    "mean_value": [],
    "approx_kl": [],
    "kl_estimate": [],
    "clip_fraction": [],
    "mean_ratio": [],
}

In [ ]:
def print_shape(name, x):
    if isinstance(x, list):
        print(f"{name}: list, length={len(x)}")
        if len(x) > 0:
            print(f"  first element shape: {np.shape(x[0])}")
    elif isinstance(x, np.ndarray):
        print(f"{name}: numpy array, shape={x.shape}")
    elif isinstance(x, torch.Tensor):
        print(f"{name}: tensor, shape={x.shape}")
    else:
        print(f"{name}: type={type(x)}")

In [ ]:
iterations = 2000
epochs_per_rollout = 10
episode_length = 1000
render_check_it = 200
render_episode = False
log_interval = 20

policy = Policy()
critic = Value()
gamma = 0.99
lam = 0.95

policy_optimizer = torch.optim.Adam(
    policy.parameters(),
    lr=3e-4
)

critic_optimizer = torch.optim.Adam(
    critic.parameters(),
    lr=3e-4
)

for i in range(iterations):
  render_episode = i % render_check_it == 0

  states, zs, actions, log_probs, rewards, dones, frames = rollout(env, policy, episode_length, render_episode)

  # episode stats
  episode_returns, episode_lengths = (
        calculate_episode_returns(rewards, dones)
    )

  if episode_returns:
    training_log["episode_return"].extend(
        episode_returns
    )

    training_log["episode_length"].extend(
        episode_lengths
    )

  states_t = torch.from_numpy(np.asarray(states)).float()
  values = critic(states_t[:-1]).squeeze(-1)
  advantages = adv(critic, states_t, rewards, dones, gamma, lam)
  returns = (values + advantages).detach()

  training_log["mean_advantage"].append(advantages.mean().item())
  training_log["std_advantage"].append(advantages.std().item())

  # Critic needs scale of advantage, policy needs stability, so normalize after calculating returns for critic
  advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

  training_log["mean_value"].append(values.mean().item())
  training_log["mean_return"].append(returns.mean().item())


  # Multiple epochs for same rollout to create divergence in policy
  for epoch in range(epochs_per_rollout):
    # policy bp
    policy_loss ,mean_ratio ,clip_fraction ,approx_kl ,kl_estimate = clipped_obj(policy,critic, states, zs, log_probs, 0.2, advantages, 0.001)
    policy_optimizer.zero_grad()
    policy_loss.backward()

    torch.nn.utils.clip_grad_norm_(policy.parameters(), 0.5)

    policy_optimizer.step()

    # Critic rollout bp
    values = critic(states_t[:-1]).squeeze(-1)
    # Logging to make sure shapes are
    # print("values shape:", values.shape)
    # print("returns shape:", returns.shape)

    critic_loss = nn.MSELoss()(values, returns)
    critic_optimizer.zero_grad()
    critic_loss.backward()

    torch.nn.utils.clip_grad_norm_(critic.parameters(), 0.5)

    critic_optimizer.step()

    # Logging
    training_log["policy_loss"].append(policy_loss.item())
    training_log["critic_loss"].append(critic_loss.item())
    training_log["mean_ratio"].append(mean_ratio)
    training_log["clip_fraction"].append(clip_fraction)
    training_log["approx_kl"].append(approx_kl)
    training_log["kl_estimate"].append(kl_estimate)

  if i % log_interval == 0:
    recent_returns = training_log["episode_return"][-log_interval:]

    if recent_returns:
      print(
          f"Iteration {i:4d} | "
          f"Return {np.mean(recent_returns):8.2f} | \n"
          f"Policy Loss {policy_loss.item():8.4f} | "
          f"Critic Loss {critic_loss.item():8.4f} | "
          f"Mean Value {values.mean().item():8.4f} | "
          f"Mean Advantage {training_log['mean_advantage'][-1]:8.4f} | "
          f"Std Advantage {training_log['std_advantage'][-1]:8.4f} | \n"
          f"KL {approx_kl:8.5f} | "
          f"KL sqr {kl_estimate:8.5f} | \n"
          f"Clip {clip_fraction:6.3f} | "
          f"Ratio {mean_ratio:6.3f}"
      )


  if render_episode:
    print(f"Rendering episode for iteration {i}")
    media.show_video(frames, fps=30)


values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([1000])
Iteration    0 | Return  1433.87 | 
Policy Loss  -0.0313 | Critic Loss  44.3286 | Mean Value  -0.0490 | Mean Advantage  -2.0096 | Std Advantage   6.4401 | 
KL  0.00843 | KL sqr  0.00992 | 
Clip  0.151 | Ratio  1.001
Rendering episode for iteration 0


This browser does not support the video tag.


values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([1000])
values shape: torch.Size([1000])
returns shape: torch.Size([10

KeyboardInterrupt: 